# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmedosrf/flyrank-ml-internship-ahmedosrf/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 2 — Refresh / Content Opportunity Scoring.** I want to identify which anonymized content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring. This lane fits the starter data because each row represents one page and includes demand, visibility, freshness, engagement, and observed movement signals. I am choosing it provisionally because the decision is concrete: an editor has limited review time, so a ranked and explainable queue is more useful than a model score without an action. I will begin with a transparent baseline and only add ML if multiple signals and their interactions improve prioritization.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data/raw/content_refresh_anonymized.csv").exists():
    ROOT = Path("/home/ubuntu/flyrank-ml-internship-starter")
DATA_PATH = ROOT / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print({"rows": len(df), "columns": len(df.columns), "clients": df["client_id"].nunique()})


{'rows': 30000, 'columns': 44, 'clients': 32}


## 2. The question: decision, action, cost of a wrong call

**Search question:** Among the pages in the current anonymized snapshot, which pages should an editor review first for a possible content refresh or protection action?

**Unit of analysis:** one pseudonymized content page at the snapshot level, with aggregated trailing-90-day metrics and recent-versus-previous-30-day movement fields.

**Output:** a ranked priority score with a suggested action and reason codes. The target, if I use classification, will be the observed `trend_direction == "down"` label; I will not use `trend_direction` or `trend_pct` as features because they define or encode that outcome. The main success metric will be Precision@K, especially Precision@50, because an editor can only act on a limited top slice.

**Decision and action:** an SEO editor or content lead uses the top of the queue to inspect pages, then decides whether to refresh, expand, protect, monitor, or leave a page unchanged. A false positive spends scarce editorial time on a page that did not need priority. A false negative can leave a genuinely declining, high-demand page unreviewed; this may cost missed traffic or delayed intervention. The notebook will support decisions, not automate publishing or guarantee that a refresh will recover traffic.

**Why data or ML can help:** a simple rule such as “stale and visible” is a necessary baseline, but the data contains several signals—demand, impressions, position, CTR, engagement, age, and freshness—that may interact. ML is justified only if it improves the ranked queue on a client-grouped holdout and remains explainable enough for a reviewer to challenge.

In [2]:
# The frame is a ranking/scoring problem, so verify the target and the decision-time fields.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
assert not {"trend_direction", "trend_pct", "content_id", "client_id"}.intersection({"search_volume", "impressions_90d", "days_since_last_update", "avg_position", "ctr"})
print({"declining_pages": int(df["is_declining_label"].sum()), "declining_rate": round(df["is_declining_label"].mean(), 3)})


{'declining_pages': 16262, 'declining_rate': np.float64(0.542)}


## 3. Quick look at the data (2-3 real numbers)

The following code loads the safe starter slice and computes the evidence for this lane. These are descriptive snapshot numbers, not causal estimates. I am especially checking the dataset grain, the amount of observed decline, and how many declining pages also have meaningful visibility for a review queue.

In [3]:
# Visibility makes a decline more actionable than a decline with no observed exposure.
visible = df["impressions_90d"] >= 500
visible_declining = visible & (df["is_declining_label"] == 1)
print({
    "visible_pages_impressions_ge_500": int(visible.sum()),
    "visible_declining_pages": int(visible_declining.sum()),
    "visible_declining_rate": round(df.loc[visible, "is_declining_label"].mean(), 3),
    "median_impressions_visible_declining": float(df.loc[visible_declining, "impressions_90d"].median()),
})


{'visible_pages_impressions_ge_500': 16726, 'visible_declining_pages': 9961, 'visible_declining_rate': np.float64(0.596), 'median_impressions_visible_declining': 2769.0}


## 4. Careful words: what I can and can't claim

I can report **observed** relationships in this anonymized snapshot, such as which pages receive higher priority under a stated scoring rule and whether a model ranks observed declining pages well on a client-grouped holdout. I can make **directional** statements about associations between safe page signals and the observed movement label. The result is **decision-support** for review prioritization, not an automated editorial decision.

I cannot claim that a feature causes rankings to change, that refreshing a page will recover traffic, or that the analysis predicts Google's algorithm. I also cannot claim generalization to every client or future time period without a proper time-aware or client-aware validation design. The final lane may change by Week 4 if the larger release or mentor feedback shows that another question is more useful.

In [4]:
# Self-check: the label is observed from trend_direction, while identifiers remain grouping keys only.
assert len(df) == 30000
assert df["client_id"].nunique() == 32
assert df["is_declining_label"].isin([0, 1]).all()
print("Self-check passed: page-level snapshot, observed label, pseudonymous IDs kept for grouping only.")


Self-check passed: page-level snapshot, observed label, pseudonymous IDs kept for grouping only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.